# Preflight de un único paso real — A2D2 + UrbanSyn

Esta notebook ejecuta una comprobación acotada por cada combinación dataset/tarea: A2D2 semántica, UrbanSyn semántica, A2D2 instancias y UrbanSyn instancias. Cada caso corre en un proceso aislado, con resolución 512×512 y batch 1. **Cada caso aprobado realiza exactamente un `backward()` y un `optimizer.step()` sobre un único batch TRAIN real.**

No ejecuta épocas, no itera validation, no abre test, no materializa ni descarga datos, no guarda checkpoints y no inicia el piloto. Los pesos son aleatorios (sin preentrenamiento); el learning rate es solo de sondeo, no una configuración de experimento.

**Por defecto la ejecución está bloqueada.** Para autorizar esta prueba limitada, cambia manualmente `EXECUTE_SINGLE_STEP = True` en la celda 5 después de revisar todas las condiciones.

## 1. Montar Drive y preparar destino de informes

Se leerán los informes de preflight y capacidad ya existentes. Los resultados se guardarán en Drive.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, json, subprocess, shutil, datetime

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/ProcesamientoImagenes')
OUT_DIR = DRIVE_ROOT / 'preflight_un_paso_real'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('Informes:', OUT_DIR)


## 2. Sincronizar la rama privada

Usa el secreto `GITHUB_TOKEN` (Contents: Read-only). No pegues tokens en el notebook ni en el chat.

In [ ]:
from getpass import getpass
REPO='Leohf04/Procesamiento-de-Im-genes-'
BRANCH='urban-models-and-benchmark'
WORKDIR=Path('/content/Procesamiento-de-Im-genes-')
try:
    from google.colab import userdata
    TOKEN=userdata.get('GITHUB_TOKEN')
except Exception:
    TOKEN=None
if not TOKEN:
    TOKEN=getpass('GitHub PAT (Contents: Read-only): ')
if not TOKEN:
    raise RuntimeError('Falta GITHUB_TOKEN.')
askpass=Path('/tmp/colab_git_askpass.sh')
askpass.write_text('#!/bin/sh\ncase "$1" in\n *Username*) echo x-access-token ;;\n *Password*) echo "$COLAB_GITHUB_TOKEN" ;;\n *) exit 1 ;;\nesac\n')
askpass.chmod(0o700)
env=os.environ.copy(); env.update({'GIT_ASKPASS':str(askpass),'GIT_TERMINAL_PROMPT':'0','COLAB_GITHUB_TOKEN':TOKEN})
if WORKDIR.exists():
    subprocess.run(['git','-C',str(WORKDIR),'fetch','origin',BRANCH],check=True,env=env)
    subprocess.run(['git','-C',str(WORKDIR),'checkout',BRANCH],check=True)
    subprocess.run(['git','-C',str(WORKDIR),'pull','--ff-only','origin',BRANCH],check=True,env=env)
else:
    subprocess.run(['git','clone','--single-branch','--branch',BRANCH,f'https://github.com/{REPO}.git',str(WORKDIR)],check=True,env=env)
del TOKEN,env
os.chdir(WORKDIR)
HEAD=subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()
print('HEAD:',HEAD)


## 3. Verificar GPU/Drive y auditoría sintética de memoria

Se exige el preflight GPU/Drive aprobado y las seis pruebas sintéticas de memoria PASS. Esta celda no ejecuta modelos ni actualiza parámetros.

In [ ]:
import torch
runtime_path=DRIVE_ROOT/'auditoria_entorno_entrenamiento'/'training_runtime_preflight.json'
memory_path=DRIVE_ROOT/'auditoria_memoria_gpu'/'gpu_memory_audit_summary.json'
if not runtime_path.is_file(): raise FileNotFoundError(f'Falta {runtime_path}; vuelve a ejecutar el preflight GPU/Drive.')
if not memory_path.is_file(): raise FileNotFoundError(f'Falta {memory_path}; vuelve a ejecutar la auditoría de memoria GPU.')
runtime=json.loads(runtime_path.read_text())
memory=json.loads(memory_path.read_text())
print('Runtime preflight:',json.dumps(runtime,indent=2)[:6000])
print('Memory audit:',json.dumps(memory,indent=2)[:6000])
if not torch.cuda.is_available(): raise RuntimeError('CUDA no está disponible en esta sesión.')
if runtime.get('gpu_status')!='PASS' or runtime.get('drive_status')!='PASS':
    raise RuntimeError('El preflight persistido no confirma GPU y Drive en PASS.')
if runtime.get('cuda_available') is not True: raise RuntimeError('El informe GPU no confirma cuda_available=true.')
experiments=memory.get('cases',[])
if not experiments or len(experiments)!=6 or any(x.get('status')!='PASS' or x.get('optimizer_steps')!=0 for x in experiments):
    raise RuntimeError('No se encuentran seis casos PASS en el informe de memoria; revisar su esquema/salida.')
print('GPU:',torch.cuda.get_device_name(0),'VRAM GiB:',round(torch.cuda.get_device_properties(0).total_memory/1024**3,2))
print('GPU/Drive y seis casos sintéticos: PASS')


## 4. Verificar datos ya materializados (sin descargar)

Se reutiliza `/content/urban_data_capacity` creado por la auditoría incremental. Si el runtime se reinició y no existe, detente y vuelve a ejecutar `notebooks/colab_auditoria_capacidad_incremental.ipynb`; esta notebook no descarga ni materializa nada.

In [ ]:
DATA_DIR=Path('/content/urban_data_capacity')
INDEX_DIR=DATA_DIR/'local_annotation_index'
mat_path=DATA_DIR/'materialization_report.json'
if not mat_path.is_file(): raise FileNotFoundError('No existe materialization_report.json en /content/urban_data_capacity. Ejecuta primero la auditoría incremental de capacidad.')
mat=json.loads(mat_path.read_text())
if mat.get('status')!='PASS' or mat.get('test_touched') is not False: raise RuntimeError('La materialización no confirma PASS y test_touched=false.')
for ds in ('a2d2','urbansyn'):
  for split in ('train','val'):
    manifest=DATA_DIR/'local_manifests'/f'{ds}_{split}.jsonl'
    ann=INDEX_DIR/f'{ds}_{split}.annotations.jsonl'
    if not manifest.is_file() or not ann.is_file(): raise FileNotFoundError(f'Falta manifiesto/índice: {ds}/{split}')
    rows=[json.loads(x) for x in manifest.read_text().splitlines() if x.strip()]
    anns=[json.loads(x) for x in ann.read_text().splitlines() if x.strip()]
    if len(rows)!=10 or len(anns)!=10 or [x['sample_id'] for x in rows]!=[x['sample_id'] for x in anns]:
      raise RuntimeError(f'{ds}/{split}: se esperaban 10 IDs alineados.')
print('Materialización existente: PASS; 10 muestras por split/dataset; no se descarga nada.')


## 5. Puerta de ejecución explícita

**ATENCIÓN:** si cambias el valor a `True`, la celda 6 ejecutará cuatro procesos aislados y cada uno hará exactamente una actualización del optimizador sobre un batch real de TRAIN (cuatro `optimizer.step()` en total si todos llegan a completarse). No es el entrenamiento piloto.

In [ ]:
EXECUTE_SINGLE_STEP = False  # Cambiar manualmente a True solo para autorizar los cuatro pasos de preflight.
if EXECUTE_SINGLE_STEP:
    print('AUTORIZADO: solo un paso por combinación; 4 procesos aislados; sin epochs/validation/test/checkpoints.')
else:
    print('BLOQUEADO POR DEFECTO. No se ejecutará ningún forward/backward/optimizer.step.')


## 6. Ejecutar el preflight acotado (solo si se autorizó en la celda 5)

Cada combinación se ejecuta en un proceso independiente. Si una falla u obtiene OOM, se conserva su informe y se continúa con las demás para aislar resultados.

In [ ]:
if not EXECUTE_SINGLE_STEP:
    print('Sin ejecución: EXECUTE_SINGLE_STEP=False.')
else:
    cases=[('a2d2','semantic'),('urbansyn','semantic'),('a2d2','instances'),('urbansyn','instances')]
    results=[]
    for ds,task in cases:
        case_dir=OUT_DIR/f'{ds}_{task}'
        case_dir.mkdir(parents=True,exist_ok=True)
        report=case_dir/'single_step_report.json'
        stdout=case_dir/'stdout.log'; stderr=case_dir/'stderr.log'
        cmd=[sys.executable,'-u','scripts/urban_single_step_preflight.py',
             '--dataset',ds,'--task',task,'--data-dir',str(DATA_DIR),
             '--annotation-index-dir',str(INDEX_DIR),'--image-size','512',
             '--output',str(report),'--execute-single-step']
        print('Ejecutando',ds,task,flush=True)
        try:
            proc=subprocess.run(cmd,cwd=WORKDIR,text=True,capture_output=True,timeout=900)
            stdout.write_text(proc.stdout or '',encoding='utf-8'); stderr.write_text(proc.stderr or '',encoding='utf-8')
            detail=json.loads(report.read_text()) if report.is_file() else {'status':'FAIL','error':'No se generó informe'}
            detail['returncode']=proc.returncode
        except subprocess.TimeoutExpired as exc:
            stdout.write_text(exc.stdout or '',encoding='utf-8'); stderr.write_text(exc.stderr or '',encoding='utf-8')
            detail={'status':'FAIL','dataset':ds,'task':task,'error':'timeout_900s','optimizer_steps':0}
        results.append(detail)
        print(json.dumps(detail,indent=2)[:5000])
    summary={'status':'PASS' if len(results)==4 and all(x.get('status')=='PASS' and x.get('optimizer_steps')==1 for x in results) else 'FAIL',
             'results':results,'optimizer_steps':sum(int(x.get('optimizer_steps',0)) for x in results),
             'training_started':False,'single_step_preflight_executed':True,
             'pilot_training_started':False,'test_split_touched':False,
             'validation_batches_iterated':0,'checkpoint_written':False,
             'dataset_downloaded_by_notebook':False,'git_head':HEAD,
             'timestamp_utc':datetime.datetime.now(datetime.timezone.utc).isoformat()}
    summary_path=OUT_DIR/'single_step_preflight_summary.json'
    summary_path.write_text(json.dumps(summary,indent=2,allow_nan=False)+'\n',encoding='utf-8')
    print('RESUMEN:',summary_path)
    print(json.dumps({k:v for k,v in summary.items() if k!='results'},indent=2))
    if summary['status']!='PASS': raise RuntimeError('Uno o más casos fallaron; no iniciar el piloto. Revisar informes y logs por caso.')


## 7. Cierre

Un PASS solo demuestra que cada combinación pudo completar un paso de optimización con el batch real seleccionado, con loss/gradientes finitos y cambio de parámetros. **No demuestra convergencia, calidad, estabilidad de varias iteraciones ni autoriza el piloto.** Si hay OOM o error, conserva los JSON/logs y no cambies batch/resolución a ciegas. Test continúa bloqueado.